# Capitulo 8 - Prazos por etapas

Execute em ordem. Este notebook e independente dos arquivos Python do computador. Nao requer instalacao de bibliotecas nem chave de API. Depois de editar uma definicao, execute sua celula novamente. Salve ou baixe sua copia.


## Definicoes


In [ ]:
"""Capítulo 8: prioridades, filtros e prazos, com dados em memória."""

from copy import deepcopy
from datetime import date


def cadastrar_tarefa(tarefas, titulo, prioridade="media", prazo=None):
    if not titulo.strip():
        return "Informe um título com pelo menos um caractere que não seja espaço."
    if prioridade not in ("baixa", "media", "alta"):
        return "Prioridade inválida. Use baixa, media ou alta."
    try:
        data_prazo = interpretar_prazo(prazo)
    except ValueError as erro:
        return str(erro)
    tarefas.append({
        "titulo": titulo,
        "concluida": False,
        "prioridade": prioridade,
        "prazo": data_prazo,
    })
    return "Tarefa cadastrada."


def concluir_tarefa(tarefas, indice):
    if indice < 0 or indice >= len(tarefas):
        return "Tarefa não encontrada."
    tarefas[indice]["concluida"] = True
    return "Tarefa concluída."


def listar_tarefas(tarefas):
    if not tarefas:
        return "Nenhuma tarefa cadastrada."
    linhas = []
    for indice, tarefa in enumerate(tarefas):
        marca = "x" if tarefa["concluida"] else " "
        linhas.append(f"{indice}: [{marca}] {tarefa['titulo']}")
    return "\n".join(linhas)


def filtrar_tarefas(tarefas, prioridade=None, concluida=None):
    if prioridade is not None and prioridade not in ("baixa", "media", "alta"):
        raise ValueError("Prioridade de filtro inválida.")
    if concluida is not None and type(concluida) is not bool:
        raise ValueError("Conclusão de filtro deve ser booleana.")
    resultado = []
    for tarefa in tarefas:
        if prioridade is not None and tarefa.get("prioridade", "media") != prioridade:
            continue
        if concluida is not None and tarefa["concluida"] != concluida:
            continue
        resultado.append(deepcopy(tarefa))
    return resultado


def interpretar_prazo(texto):
    if texto is None:
        return None
    if not isinstance(texto, str):
        raise ValueError("Prazo deve ser texto no formato AAAA-MM-DD.")
    try:
        prazo = date.fromisoformat(texto)
    except ValueError:
        raise ValueError("Prazo inválido. Use AAAA-MM-DD.") from None
    if prazo.isoformat() != texto:
        raise ValueError("Prazo inválido. Use AAAA-MM-DD.")
    return prazo


def esta_atrasada(tarefa, hoje):
    prazo = tarefa.get("prazo")
    return (
        not tarefa["concluida"]
        and prazo is not None
        and prazo < hoje
    )


## Demonstracao


In [ ]:
tarefas = []
print(cadastrar_tarefa(tarefas, "Revisar", "alta", "2026-10-14"))
print(cadastrar_tarefa(tarefas, "Sem prazo"))
print(cadastrar_tarefa(tarefas, "Data impossivel", prazo="2026-02-30"))
print(listar_tarefas(tarefas))


## Conferencia dos casos

Uma falha pode produzir AssertionError. O resultado verifica estes casos, sem prometer ausencia de outros defeitos.


In [ ]:
assert len(tarefas) == 2
assert tarefas[0]["prazo"] == date(2026, 10, 14)
assert tarefas[1]["prazo"] is None
hoje = date(2026, 10, 14)
for prazo, concluida, esperado in ((date(2026,10,13), False, True), (hoje, False, False), (date(2026,10,15), False, False), (date(2026,10,13), True, False), (None, False, False)):
    assert esta_atrasada({"prazo": prazo, "concluida": concluida}, hoje) is esperado
print("Prazos e cinco casos de atraso conferidos.")
